In [0]:
# Load Silver Layer Data
df_silver = spark.table("workspace.ckd_project.ckd_silver")

display(df_silver.limit(10))

bp_diastolic,bp_limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,1.02,1,ckd,0,0,0,0,0,112.0,48.1,140.5,3.65,7.31,11.95,35.45,4.755,8550.0,0,0,0,0,0,0,227.944,s1,1,12
0,0,1.0099999999999998,0,ckd,0,0,0,0,0,133.0,48.1,135.5,3.65,7.31,11.95,35.45,4.755,13310.0,0,0,0,0,0,0,227.944,s1,1,12
0,0,1.0099999999999998,4,ckd,1,0,1,0,1,112.0,67.15,135.5,3.65,7.31,9.35,31.55,4.755,15690.0,0,0,0,1,0,0,139.8635,s1,1,12
1,1,1.0099999999999998,3,ckd,0,0,0,0,0,133.0,48.1,135.5,3.65,7.31,14.55,43.25,4.755,8550.0,0,0,0,0,0,0,139.8635,s1,1,12
0,0,1.016,0,ckd,0,0,0,0,0,175.0,48.1,135.5,3.65,7.31,14.55,39.349999999999994,5.345,8550.0,0,1,0,1,1,0,139.8635,s1,1,16
1,1,1.023,0,notckd,0,0,0,0,0,112.0,48.1,135.5,3.65,7.31,16.5,49.1,5.345,6170.0,0,0,0,0,0,0,114.69800000000001,s1,0,16
0,0,1.02,3,ckd,0,0,0,0,0,112.0,48.1,140.5,3.65,7.31,10.65,31.55,3.575,8550.0,1,1,0,0,0,0,190.195,s1,1,16
0,0,1.02,0,ckd,0,0,0,0,0,133.0,67.15,135.5,3.65,7.31,11.95,39.349999999999994,4.755,6170.0,0,0,0,0,0,0,39.20035,s4,1,16
0,0,1.023,0,notckd,0,0,0,0,0,133.0,67.15,135.5,3.65,7.31,14.55,39.349999999999994,5.345,4980.0,0,0,0,0,0,0,39.20035,s4,0,23
1,2,1.0099999999999998,4,ckd,0,0,1,1,1,112.0,48.1,125.5,3.65,7.31,8.05,23.75,4.165,13310.0,0,0,0,0,0,1,64.3661,s3,1,23


In [0]:
from pyspark.sql.functions import col, when

df_gold = df_silver \
    .withColumn("target", when(col("affected") == 1, 1).otherwise(0)) \
    .withColumn("hypertension_flag", when(col("htn") == 1, 1).otherwise(0)) \
    .withColumn("diabetes_flag", when(col("dm") == 1, 1).otherwise(0)) \
    .withColumn("heart_disease_flag", when(col("cad") == 1, 1).otherwise(0)) \
    .withColumn("edema_flag", when(col("pe") == 1, 1).otherwise(0)) \
    .withColumn("anemia_flag", when(col("ane") == 1, 1).otherwise(0))

In [0]:

df_gold = df_gold.withColumn(
    "kidney_function_flag",
    when((col("grf") < 60) | (col("sc") > 1.2) | (col("bu") > 20), 1).otherwise(0)
)

In [0]:

df_gold = df_gold.withColumn(
    "anemia_flag",
    when(
        (col("hemo") < 12) | (col("pcv") < 33) | (col("rbcc") < 4.2),
        1
    ).otherwise(0)
)

In [0]:
from pyspark.sql.functions import col, when

df_gold = df_gold.withColumn(
    "bp_flag",
    when((col("bp_diastolic") >= 90) | (col("bp_limit") >= 140), 1).otherwise(0)
)

In [0]:
df_gold = df_gold.withColumn(
    "stage_group",
    when(col("stage").isin("s1", "s2"), "early")
    .when(col("stage").isin("s3", "s4"), "moderate")
    .when(col("stage").isin("s5"), "severe")
    .otherwise("unknown")
)


In [0]:
df_gold = df_gold.withColumn(
    "risk_score",
    col("target") +
    col("hypertension_flag") +
    col("diabetes_flag") +
    col("heart_disease_flag") +
    col("edema_flag") +
    col("anemia_flag") +
    col("kidney_function_flag") +
    col("bp_flag")
)

In [0]:
df_gold = df_gold.withColumn(
    "risk_band",
    when(col("risk_score") <= 2, "low")
    .when((col("risk_score") >= 3) & (col("risk_score") <= 4), "medium")
    .when(col("risk_score") >= 5, "high")
    .otherwise("unknown")
)

In [0]:
df_gold = df_gold.withColumn(
    "age_band",
    when(col("age") < 30, "under_30")
    .when((col("age") >= 30) & (col("age") < 45), "30_44")
    .when((col("age") >= 45) & (col("age") < 60), "45_59")
    .when((col("age") >= 60) & (col("age") < 75), "60_74")
    .when(col("age") >= 75, "75_plus")
    .otherwise("unknown")
)

In [0]:
#selecting only the golf colums for intelligence 
gold_columns = [
    "target", "risk_score", "risk_band", "stage_group", "age_band",
    "hypertension_flag", "diabetes_flag", "heart_disease_flag",
    "edema_flag", "anemia_flag", "kidney_function_flag", "bp_flag"
]

df_final_gold = df_gold.select(*gold_columns)

In [0]:
display(df_final_gold.limit(10))
# Write Gold Layer Data
df_final_gold.write.mode("overwrite").saveAsTable("ckd_gold")

target,risk_score,risk_band,stage_group,age_band,hypertension_flag,diabetes_flag,heart_disease_flag,edema_flag,anemia_flag,kidney_function_flag,bp_flag
1,3,medium,early,under_30,0,0,0,0,1,1,0
1,3,medium,early,under_30,0,0,0,0,1,1,0
1,3,medium,early,under_30,0,0,0,0,1,1,0
1,2,low,early,under_30,0,0,0,0,0,1,0
1,4,medium,early,under_30,0,1,0,1,0,1,0
0,1,low,early,under_30,0,0,0,0,0,1,0
1,5,high,early,under_30,1,1,0,0,1,1,0
1,3,medium,moderate,under_30,0,0,0,0,1,1,0
0,1,low,moderate,under_30,0,0,0,0,0,1,0
1,3,medium,moderate,under_30,0,0,0,0,1,1,0
